# Visual Module


In [ ]:
import os
import gc
import re
import time
import random
import inspect
import warnings
import contextlib
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import spacy
import kagglehub

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor, AutoProcessor, AutoModelForZeroShotObjectDetection


In [ ]:
warnings.filterwarnings("ignore")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

dataset_csv = "dataset_df.csv"
output_dir = "visual_module_outputs"
features_dir = os.path.join(output_dir, "feature_parts")
features_path = os.path.join(output_dir, "visual_features_all.csv")

row_limit = -1
clip_batch_size = 64
dino_batch_size = 1
chunk_size = 100

max_dino_text_tokens = 256
max_dino_phrases_per_image = 50
dino_box_threshold = 0.25
dino_text_threshold = 0.25
bad_phrase_threshold = 0.10

clip_model_name = "openai/clip-vit-base-patch32"
dino_model_name = "IDEA-Research/grounding-dino-base"
sentence_model_name = "sentence-transformers/all-MiniLM-L6-v2"

os.makedirs(output_dir, exist_ok = True)
os.makedirs(features_dir, exist_ok = True)

## Загрузка и проверка данных

In [ ]:
def numeric_suffix(column_name):
    match = re.search(r"_(\d+)$", column_name)
    return int(match.group(1)) if match is not None else 10 ** 9


df_dataset = pd.read_csv(dataset_csv)

caption_columns = sorted([column for column in df_dataset.columns if re.match(r"generated_caption_\d+$", column)], key = numeric_suffix)
real_caption_columns = sorted([column for column in df_dataset.columns if re.match(r"real_caption_\d+$", column)], key = numeric_suffix)

for column in caption_columns + real_caption_columns:
    df_dataset[column] = df_dataset[column].fillna("").astype(str)

df_dataset["file_path"] = df_dataset["file_path"].astype(str)
exists_mask = df_dataset["file_path"].apply(os.path.exists)

if not exists_mask.all():
    dataset_path = kagglehub.dataset_download("nikhil7280/coco-image-caption")
    file_to_path = {}
    for root, dirs, files in os.walk(dataset_path):
        for file_name in files:
            if file_name.endswith(".jpg"):
                file_to_path[file_name] = os.path.join(root, file_name)

    file_names = df_dataset["file_path"].apply(os.path.basename)
    resolved_paths = file_names.map(file_to_path)
    df_dataset.loc[~exists_mask, "file_path"] = resolved_paths[~exists_mask]

exists_mask = df_dataset["file_path"].notna() & df_dataset["file_path"].apply(os.path.exists)
df_dataset = df_dataset[exists_mask].copy().reset_index(drop = True)

if row_limit > 0:
    df_dataset = df_dataset.iloc[:row_limit].copy().reset_index(drop = True)

df_dataset["row_index"] = np.arange(len(df_dataset))

print("Количество изображений ", len(df_dataset))
print("Количество заголовков", len(caption_columns))
print("Количество заголовков COCO", len(real_caption_columns))


## Загрузка моделей


In [ ]:
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()

dino_processor = AutoProcessor.from_pretrained(dino_model_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_model_name).to(device).eval()

sentence_model = SentenceTransformer(sentence_model_name, device = device)
nlp = spacy.load("en_core_web_sm")

## Обработка текстовых фраз и очистка кеша

In [ ]:
stop_phrases = {"a", "an", "the", "this", "that", "it", "there", "image", "picture", "photo"}


def amp_context():
    if device == "cuda":
        return torch.autocast("cuda", dtype = amp_dtype)
    return contextlib.nullcontext()


def clear_cache():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def normalize_phrase(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_iou(first_text, second_text):
    first_tokens = set(normalize_phrase(first_text).split())
    second_tokens = set(normalize_phrase(second_text).split())

    if len(first_tokens) == 0 or len(second_tokens) == 0:
        return 0.0

    return len(first_tokens & second_tokens) / len(first_tokens | second_tokens)


def get_tensor(output):
    if isinstance(output, torch.Tensor):
        return output

    if hasattr(output, "image_embeds"):
        return output.image_embeds

    if hasattr(output, "text_embeds"):
        return output.text_embeds

    if hasattr(output, "pooler_output"):
        return output.pooler_output

    if hasattr(output, "last_hidden_state"):
        return output.last_hidden_state[:, 0, :]

    return output[0]


def extract_phrases(caption):
    doc = nlp(str(caption).lower())
    phrases = []

    for chunk in doc.noun_chunks:
        phrase = normalize_phrase(chunk.text)

        if len(phrase) > 1 and phrase not in stop_phrases:
            phrases.append(phrase)

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            phrase = normalize_phrase(token.lemma_)

            if len(phrase) > 1 and phrase not in stop_phrases:
                phrases.append(phrase)

    return list(dict.fromkeys(phrases))


def collect_phrases(captions):
    counter = Counter()

    for caption in captions:
        for phrase in extract_phrases(caption):
            counter[phrase] += 1

    return [phrase for phrase, count in counter.most_common(max_dino_phrases_per_image)]


## Оценка соответствия изображения и подписи через CLIP

In [ ]:
def compute_clip_scores(image_paths, captions_per_image):
    images = [Image.open(path).convert("RGB") for path in image_paths]
    all_captions = []
    blocks = []

    for captions in captions_per_image:
        start = len(all_captions)
        captions = [str(caption) for caption in captions]
        all_captions.extend(captions)
        blocks.append((start, len(captions)))

    image_inputs = clip_processor(images = images, return_tensors = "pt", padding = True)
    text_inputs = clip_processor(text = all_captions, return_tensors = "pt", padding = True, truncation = True)
    image_inputs = {name: value.to(device) for name, value in image_inputs.items()}
    text_inputs = {name: value.to(device) for name, value in text_inputs.items()}

    with torch.inference_mode(), amp_context():
        image_features = clip_model.get_image_features(**image_inputs)
        text_features = clip_model.get_text_features(**text_inputs)

    image_features = F.normalize(get_tensor(image_features).float(), dim = -1)
    text_features = F.normalize(get_tensor(text_features).float(), dim = -1)
    scores_per_image = []

    for image_index, (start, count) in enumerate(blocks):
        scores = image_features[image_index:image_index + 1] @ text_features[start:start + count].T
        scores_per_image.append(scores.squeeze(0).detach().cpu().numpy())

    return scores_per_image


def compute_coco_similarity(generated_per_image, real_per_image):
    all_texts = []
    blocks = []

    for generated_captions, real_captions in zip(generated_per_image, real_per_image):
        generated_captions = [str(text) for text in generated_captions]
        real_captions = [str(text) for text in real_captions]
        start = len(all_texts)
        all_texts.extend(generated_captions)
        all_texts.extend(real_captions)
        blocks.append((start, len(generated_captions), len(real_captions)))

    embeddings = sentence_model.encode(all_texts, convert_to_tensor = True, normalize_embeddings = True, batch_size = 512, show_progress_bar = False)
    similarities_per_image = []

    for start, generated_count, real_count in blocks:
        generated_embeddings = embeddings[start:start + generated_count]
        real_embeddings = embeddings[start + generated_count:start + generated_count + real_count]
        similarities = generated_embeddings @ real_embeddings.T
        similarities = similarities.max(dim = 1).values.detach().cpu().numpy()
        similarities_per_image.append(similarities)

    return similarities_per_image

## GroundingDINO


In [ ]:
def dino_token_count(text):
    encoded = dino_processor.tokenizer(str(text), add_special_tokens = True, truncation = False, return_attention_mask = False)
    return len(encoded["input_ids"])


def build_dino_prompt(phrases):
    phrases = [normalize_phrase(phrase) for phrase in phrases]
    phrases = [phrase for phrase in phrases if len(phrase) > 0]
    phrases = list(dict.fromkeys(phrases))
    selected = []

    for phrase in phrases:
        prompt = ". ".join(selected + [phrase]) + "."

        if dino_token_count(prompt) <= max_dino_text_tokens:
            selected.append(phrase)

    if len(selected) == 0:
        return "object.", []

    return ". ".join(selected) + ".", selected


def match_dino_labels(labels, scores, phrases):
    phrase_scores = {phrase: 0.0 for phrase in phrases}

    for label, score in zip(labels, scores):
        label = normalize_phrase(label)
        score = float(score)

        for phrase in phrases:
            overlap = token_iou(label, phrase)

            if label == phrase or overlap >= 0.5 or label in phrase or phrase in label:
                phrase_scores[phrase] = max(phrase_scores[phrase], score)

    return phrase_scores


def postprocess_dino(outputs, inputs, images):
    target_sizes = torch.tensor([[image.height, image.width] for image in images], device = device)
    postprocess_fn = dino_processor.post_process_grounded_object_detection
    params = inspect.signature(postprocess_fn).parameters
    kwargs = {}

    if "input_ids" in params:
        kwargs["input_ids"] = inputs["input_ids"]

    if "target_sizes" in params:
        kwargs["target_sizes"] = target_sizes

    if "box_threshold" in params:
        kwargs["box_threshold"] = dino_box_threshold

    if "threshold" in params:
        kwargs["threshold"] = dino_box_threshold

    if "text_threshold" in params:
        kwargs["text_threshold"] = dino_text_threshold

    return postprocess_fn(outputs, **kwargs)


def compute_dino_scores(image_paths, phrases_per_image):
    all_phrase_scores = []

    for start in range(0, len(image_paths), dino_batch_size):
        batch_paths = image_paths[start:start + dino_batch_size]
        batch_phrases = phrases_per_image[start:start + dino_batch_size]
        images = [Image.open(path).convert("RGB") for path in batch_paths]
        prompts = []
        clean_phrases = []

        for phrases in batch_phrases:
            prompt, selected = build_dino_prompt(phrases)
            prompts.append(prompt)
            clean_phrases.append(selected)

        inputs = dino_processor(images = images, text = prompts, return_tensors = "pt", padding = "max_length", truncation = True, max_length = max_dino_text_tokens)
        inputs = {name: value.to(device) for name, value in inputs.items()}

        with torch.inference_mode(), amp_context():
            outputs = dino_model(**inputs)

        results = postprocess_dino(outputs, inputs, images)

        for result, phrases in zip(results, clean_phrases):
            labels = result.get("labels", result.get("text_labels", []))
            scores = result.get("scores", [])

            if isinstance(scores, torch.Tensor):
                scores = scores.detach().cpu().numpy()

            all_phrase_scores.append(match_dino_labels(labels, scores, phrases))

        del images, inputs, outputs, results
        clear_cache()

    return all_phrase_scores


def summarize_phrase_scores(phrase_scores):
    scores = list(phrase_scores.values())

    if len(scores) == 0:
        return {"phrase_count": 0, "grounded_count": 0, "missed_count": 0, "grounded_ratio": 0.0, "mean_ground_score": 0.0, "min_ground_score": 0.0, "max_ground_score": 0.0, "has_bad_phrase": 0.0}

    grounded_count = int(sum(score >= bad_phrase_threshold for score in scores))

    return {
        "phrase_count": int(len(scores)),
        "grounded_count": grounded_count,
        "missed_count": int(len(scores) - grounded_count),
        "grounded_ratio": float(grounded_count / len(scores)),
        "mean_ground_score": float(np.mean(scores)),
        "min_ground_score": float(np.min(scores)),
        "max_ground_score": float(np.max(scores)),
        "has_bad_phrase": float(np.min(scores) < bad_phrase_threshold)
    }


## Формирование строк признаков

In [ ]:
def get_finished_rows():
    finished_rows = set()

    if os.path.exists(features_path):
        old_df = pd.read_csv(features_path, usecols = ["row_index"])
        finished_rows.update(old_df["row_index"].unique().tolist())

    part_files = sorted([os.path.join(features_dir, file_name) for file_name in os.listdir(features_dir) if file_name.startswith("features_part_") and file_name.endswith(".csv")])

    for path in part_files:
        part_df = pd.read_csv(path, usecols = ["row_index"])
        finished_rows.update(part_df["row_index"].unique().tolist())

    return finished_rows


def save_part(rows, part_id):
    part_df = pd.DataFrame(rows)
    path = os.path.join(features_dir, f"features_part_{part_id:06d}.csv")
    part_df.to_csv(path, index = False)
    print(f"saved {path}: {len(part_df)} rows")


def make_feature_rows(batch_df):
    image_paths = batch_df["file_path"].tolist()
    generated_per_image = batch_df[caption_columns].values.tolist()
    real_per_image = batch_df[real_caption_columns].values.tolist()

    clip_scores = compute_clip_scores(image_paths, generated_per_image)
    coco_scores = compute_coco_similarity(generated_per_image, real_per_image)
    phrases_per_image = [collect_phrases(captions) for captions in generated_per_image]
    dino_scores = compute_dino_scores(image_paths, phrases_per_image)
    rows = []

    for image_index, row in batch_df.reset_index(drop = True).iterrows():
        row_index = int(row["row_index"])
        image_path = row["file_path"]

        for caption_index, caption in enumerate(generated_per_image[image_index]):
            caption_phrases = extract_phrases(caption)
            caption_phrase_scores = {}

            for phrase in caption_phrases:
                phrase = normalize_phrase(phrase)
                caption_phrase_scores[phrase] = float(dino_scores[image_index].get(phrase, 0.0))

            phrase_summary = summarize_phrase_scores(caption_phrase_scores)

            rows.append({
                "row_index": row_index,
                "file_path": image_path,
                "caption_index": caption_index + 1,
                "generated_caption": str(caption),
                "clip_score": float(clip_scores[image_index][caption_index]),
                "phrase_count": phrase_summary["phrase_count"],
                "grounded_count": phrase_summary["grounded_count"],
                "missed_count": phrase_summary["missed_count"],
                "grounded_ratio": phrase_summary["grounded_ratio"],
                "mean_ground_score": phrase_summary["mean_ground_score"],
                "min_ground_score": phrase_summary["min_ground_score"],
                "max_ground_score": phrase_summary["max_ground_score"],
                "has_bad_phrase": phrase_summary["has_bad_phrase"],
                "coco_similarity": float(coco_scores[image_index][caption_index])
            })

    return rows


## Расчёт визуальных признаков

In [ ]:
finished_rows = get_finished_rows()
pending_df = df_dataset[~df_dataset["row_index"].isin(finished_rows)].copy().reset_index(drop = True)
part_id = len([file_name for file_name in os.listdir(features_dir) if file_name.startswith("features_part_") and file_name.endswith(".csv")])


for chunk_start in range(0, len(pending_df), chunk_size):
    chunk_df = pending_df.iloc[chunk_start:chunk_start + chunk_size].copy().reset_index(drop = True)
    chunk_rows = []
    for batch_start in range(0, len(chunk_df), clip_batch_size):
        batch_df = chunk_df.iloc[batch_start:batch_start + clip_batch_size].copy().reset_index(drop = True)
        batch_rows = make_feature_rows(batch_df)
        chunk_rows.extend(batch_rows)
        done = len(finished_rows) + chunk_start + batch_start + len(batch_df)
        print(f"processed {done}/{len(df_dataset)}")

    save_part(chunk_rows, part_id)
    part_id += 1
    clear_cache()


## Объединение признаков

In [ ]:
feature_files = sorted([os.path.join(features_dir, file_name) for file_name in os.listdir(features_dir) if file_name.startswith("features_part_") and file_name.endswith(".csv")])
visual_df = pd.concat([pd.read_csv(path) for path in feature_files], ignore_index = True)
visual_df = visual_df.drop_duplicates(subset = ["row_index", "caption_index"]).reset_index(drop = True)
visual_df.to_csv(features_path, index = False)

print("Размер итоговой таблицы признаков:", visual_df.shape)
print("Итоговый файл с признаками:", features_path)
print(visual_df.head())

## Подготовка данных для RankNet

In [ ]:
feature_columns = ["clip_score", "phrase_count", "grounded_count", "missed_count", "grounded_ratio", "mean_ground_score", "min_ground_score", "max_ground_score", "has_bad_phrase"]

pair_min_delta = 0.02
max_pairs_per_image = 24
ranker_batch_size = 32768
ranker_epochs = 30
ranker_patience = 5
ranker_lr = 1e-3

unique_rows = visual_df["row_index"].unique()
train_row_indices, valid_row_indices = train_test_split(unique_rows, test_size = 0.2, random_state = seed)

train_rows_set = set(train_row_indices)
valid_rows_set = set(valid_row_indices)

train_mask = visual_df["row_index"].isin(train_rows_set).to_numpy()
valid_mask = visual_df["row_index"].isin(valid_rows_set).to_numpy()

train_mean = visual_df.loc[train_mask, feature_columns].mean()
train_std = visual_df.loc[train_mask, feature_columns].std().replace(0, 1).fillna(1)

all_features = visual_df[feature_columns].to_numpy(np.float32)
all_features = (all_features - train_mean.to_numpy(np.float32)) / train_std.to_numpy(np.float32)
all_quality = visual_df["coco_similarity"].to_numpy(np.float32)
all_rows = visual_df["row_index"].to_numpy()


def make_rank_pairs(mask, name):
    rows = all_rows[mask]
    features = all_features[mask]
    quality = all_quality[mask]

    order = np.argsort(rows, kind = "mergesort")
    rows = rows[order]
    features = features[order]
    quality = quality[order]

    split_points = np.flatnonzero(rows[1:] != rows[:-1]) + 1
    starts = np.r_[0, split_points]
    ends = np.r_[split_points, len(rows)]
    better_list = []
    worse_list = []

    for start, end in zip(starts, ends):
        q = quality[start:end]
        x = features[start:end]

        order = np.argsort(-q)
        q = q[order]
        x = x[order]
        pairs = []

        for better_index in range(len(q)):
            for worse_index in range(len(q) - 1, -1, -1):
                if q[better_index] - q[worse_index] < pair_min_delta:
                    break

                pairs.append((better_index, worse_index))

        if len(pairs) > max_pairs_per_image:
            pair_indices = np.random.choice(len(pairs), max_pairs_per_image, replace = False)
            pairs = [pairs[index] for index in pair_indices]

        for better_index, worse_index in pairs:
            better_list.append(x[better_index])
            worse_list.append(x[worse_index])

    better_array = np.asarray(better_list, dtype = np.float32)
    worse_array = np.asarray(worse_list, dtype = np.float32)

    print(f"{name} pairs:", len(better_array))

    return better_array, worse_array


train_better, train_worse = make_rank_pairs(train_mask, "train")
valid_better, valid_worse = make_rank_pairs(valid_mask, "valid")


## Загрузчики обучающих пар

In [ ]:
class RankPairDataset(Dataset):
    def __init__(self, better_features, worse_features):
        self.better_features = torch.tensor(better_features, dtype = torch.float32)
        self.worse_features = torch.tensor(worse_features, dtype = torch.float32)

    def __len__(self):
        return len(self.better_features)

    def __getitem__(self, index):
        return self.better_features[index], self.worse_features[index]


train_loader = DataLoader(RankPairDataset(train_better, train_worse), batch_size = ranker_batch_size, shuffle = True)
valid_loader = DataLoader(RankPairDataset(valid_better, valid_worse), batch_size = ranker_batch_size, shuffle = False)

## Обучение RankNet

In [ ]:
ranker = nn.Sequential(nn.Linear(len(feature_columns), 64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.10), nn.Linear(32, 1)).to(device)
optimizer = torch.optim.AdamW(ranker.parameters(), lr = ranker_lr, weight_decay = 1e-4)

best_valid_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(ranker_epochs):
    ranker.train()
    train_loss_sum = 0.0

    for better_features, worse_features in train_loader:
        better_features = better_features.to(device)
        worse_features = worse_features.to(device)

        optimizer.zero_grad(set_to_none = True)
        better_score = ranker(better_features).squeeze(1)
        worse_score = ranker(worse_features).squeeze(1)
        loss = -F.logsigmoid(better_score - worse_score).mean()

        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()

    ranker.eval()
    valid_loss_sum = 0.0

    with torch.inference_mode():
        for better_features, worse_features in valid_loader:
            better_features = better_features.to(device)
            worse_features = worse_features.to(device)
            better_score = ranker(better_features).squeeze(1)
            worse_score = ranker(worse_features).squeeze(1)
            loss = -F.logsigmoid(better_score - worse_score).mean()
            valid_loss_sum += loss.item()

    train_loss = train_loss_sum / max(1, len(train_loader))
    valid_loss = valid_loss_sum / max(1, len(valid_loader))
    print(f"epoch {epoch + 1}: train_loss={train_loss:.5f}, valid_loss={valid_loss:.5f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {name: value.detach().cpu().clone() for name, value in ranker.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= ranker_patience:
        print("early stopping")
        break

if best_state is not None:
    ranker.load_state_dict(best_state)


## Проверка качества выбора подписи

In [ ]:
def add_ranker_scores(dataframe):
    ranker.eval()

    mean = train_mean.to_numpy(np.float32)
    std = train_std.to_numpy(np.float32)
    features = dataframe[feature_columns].to_numpy(np.float32)
    features = (features - mean) / std
    scores = []

    for start in range(0, len(features), 65536):
        batch = torch.tensor(features[start:start + 65536], dtype = torch.float32, device = device)

        with torch.inference_mode():
            batch_scores = ranker(batch).squeeze(1).detach().cpu().numpy()

        scores.extend(batch_scores.tolist())

    result_df = dataframe.copy()
    result_df["ranker_score"] = scores

    return result_df


valid_df = visual_df[valid_mask].copy().reset_index(drop = True)
valid_scored_df = add_ranker_scores(valid_df)

baseline_df = valid_scored_df[valid_scored_df["caption_index"] == 1].copy().reset_index(drop = True)
clip_selected_df = valid_scored_df.sort_values(["row_index", "clip_score"], ascending = [True, False]).groupby("row_index").head(1).reset_index(drop = True)
ranknet_selected_df = valid_scored_df.sort_values(["row_index", "ranker_score"], ascending = [True, False]).groupby("row_index").head(1).reset_index(drop = True)
oracle_df = valid_scored_df.sort_values(["row_index", "coco_similarity"], ascending = [True, False]).groupby("row_index").head(1).reset_index(drop = True)

summary_df = pd.DataFrame([
    {"method": "baseline_first_blip", "mean_coco_similarity": baseline_df["coco_similarity"].mean()},
    {"method": "clip_selected", "mean_coco_similarity": clip_selected_df["coco_similarity"].mean()},
    {"method": "ranknet_selected", "mean_coco_similarity": ranknet_selected_df["coco_similarity"].mean()},
    {"method": "oracle_best_among_candidates", "mean_coco_similarity": oracle_df["coco_similarity"].mean()}
])

summary_path = os.path.join(output_dir, "visual_module_summary.csv")
ranknet_selected_path = os.path.join(output_dir, "ranknet_selected_captions.csv")
clip_selected_path = os.path.join(output_dir, "clip_selected_captions.csv")

summary_df.to_csv(summary_path, index = False)
ranknet_selected_df.to_csv(ranknet_selected_path, index = False)
clip_selected_df.to_csv(clip_selected_path, index = False)

print(summary_df)
print("saved:", summary_path)
print("saved:", ranknet_selected_path)
print("saved:", clip_selected_path)


## Попарная проверка ранжирования

In [ ]:
def compute_pair_accuracy(scored_df, score_column, min_delta = pair_min_delta):
    correct = 0
    total = 0

    for row_index, group in scored_df.groupby("row_index"):
        group = group.reset_index(drop = True)

        for first_index in range(len(group)):
            for second_index in range(len(group)):
                first_quality = float(group.loc[first_index, "coco_similarity"])
                second_quality = float(group.loc[second_index, "coco_similarity"])

                if first_quality - second_quality >= min_delta:
                    first_score = float(group.loc[first_index, score_column])
                    second_score = float(group.loc[second_index, score_column])

                    if first_score > second_score:
                        correct += 1

                    total += 1

    return correct / total if total > 0 else 0.0


ranknet_pair_accuracy = compute_pair_accuracy(valid_scored_df, "ranker_score")
clip_pair_accuracy = compute_pairwise_accuracy(valid_scored_df, "clip_score")

print(f"Ranknet pair accuracy: {ranknet_pair_accuracy:.4f}")
print(f"CLIP pair accuracy: {clip_pair_accuracy:.4f}")


## Сохранение обученной модели


In [ ]:
model_path = os.path.join(output_dir, "visual_ranknet_model.pt")

model_data = {
    "ranker_state_dict": ranker.state_dict(),
    "feature_columns": feature_columns,
    "train_mean": train_mean.to_dict(),
    "train_std": train_std.to_dict(),
    "config": {
        "dino_box_threshold": dino_box_threshold,
        "dino_text_threshold": dino_text_threshold,
        "pair_min_delta": pair_min_delta,
        "max_pairs_per_image": max_pairs_per_image,
        "ranker_batch_size": ranker_batch_size,
        "ranker_epochs": ranker_epochs,
        "ranker_patience": ranker_patience,
        "ranker_lr": ranker_lr,
        "seed": seed
    }
}

torch.save(model_data, model_path)
print("saved:", model_path)

In [ ]:
preview_columns = ["row_index", "caption_index", "generated_caption", "clip_score", "ranker_score", "coco_similarity", "phrase_count", "grounded_count", "missed_count"]
print(ranknet_selected_df[preview_columns].head(20))